Import Library that we need

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, recall_score, precision_score, roc_auc_score

we use pandas as pd to load the data with its function read_csv('Path of the File
')

In [ ]:
try:
  data = pd.read_csv('/content/Telco-Customer-Churn.csv')
  print(f'Successfully loaded the Data')
  data.head()
except Exception as e:
  print(f'Data Loading failed: {e}')
  exit()

Successfully loaded the Data


After Data Loading let see the data if everything is fine then we are good to go but in our case we do have. Different Data Type at 19th index 'TotalCharges' as it is numeric data but the data type is shown as Object so we do have to take care of this.

In [ ]:
print(f'The shape of the Data is : {data.shape}')
data.info()

The shape of the Data is : (7043, 21)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  Pape

To change a column we use to_numeric(data['column name']) function from pandas library. As thers could be error so we have to also handle it also remove NA values from it. When removing NA Values we have to make sure that we are not removing large quantity as it could affect our model training


In [ ]:
data['TotalCharges'] = pd.to_numeric(data['TotalCharges'], errors='coerce')
original_rows = len(data)
data= data.dropna(subset=['TotalCharges'])
print(f'Removed rows {original_rows-len(data)}')

Removed rows 11


we can use .colmns attribute to see the column names as this information could help use to divide our data.

In [ ]:
data.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

In [ ]:
numeric_features = ['tenure','MonthlyCharges', 'TotalCharges']
target_feature ='Churn'
categorical_features = ['gender', 'SeniorCitizen', 'Partner', 'Dependents','PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport','StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod']



Once we divide our data as per our requirment now we have to differentiate between independent and dependent values. In our case we use x as independent values and y as dependent value.

In [ ]:
x=data[numeric_features+categorical_features]
y=data[target_feature]

As machines know only 0 and 1 so our targeting value should be in numeric form. for this map function to Yes as 1 and No as 0

In [ ]:
y=y.map({'Yes':1,'No':0})

Now we have Independent and Dependent values we can use train_test_split function from sklearn.model_selection to split our data in training and testing. test_size indicate how much data we will use for test,random_state we can use any value or None using same value make sure that our data will split same evertime for example if i use 42 and other person use 10 then our split data will be different due to this trained model result also vary, stratify = y means that data ration should we same for example if we have Total 100 values out of which 40 values state 'yes' and 60 values 'No' it means data is 2:3 ratio. so when our data will be split for training it make sure that data stays in same ratio.

In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42, stratify = y)


Now we can process our data. To process it we can build pipline. in pipeline we do have add steps means what we want to apply on our data like imputer if there is any null value in numeric data we will replaceit with median and in categorical data we can replace it with mode or we can say most_frequent.
To scale Numeric data we use StandardScaler() but for categorical data we have to use OneHotEncoder it is not scaler as we know machine only understand 0 and 1 it will help categorical data to 0 and 1.
Once we done with setting rule then we can apply that rule to the columns with the help of ColumnTransformer() after that we can creat final pipeline in which we will use processed data and a model which we want to train.

In [ ]:
from sklearn.compose import ColumnTransformer
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder())
])
preprocessor = ColumnTransformer(transformers=[
    ('num',numeric_transformer, numeric_features ),
    ('cat', categorical_transformer, categorical_features)
])

clf_pipeline =Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced',random_state=42))
])




Everything is ready now we just have to training data to our pipeline


In [ ]:
clf_pipeline.fit(x_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['tenure', 'MonthlyCharges',
                                                   'TotalCharges']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder())]),
                                                  ['gender', 'SeniorCitizen',
                                                   'Partner', 'Dependents',
                                                   'PhoneService',
                                                   'MultipleLines',
                                                   'InternetService',
                                                   'OnlineSecurity',
                                                   'OnlineBackup',
                                                   'DeviceProtection',
                                                   'TechSupport', 'StreamingTV',
                                                   'StreamingMovies',
                                                   'Contract',
                                                   'PaperlessBilling',
                                                   'PaymentMethod'])])),
                ('classifier',
                 LogisticRegression(class_weight='balanced', random_state=42))])

Now we have successfully trained our LogisticRegression model. we can't repeat same steps every time when we want to use our model. so we use pickle library which help us to dump our trained model so we can use pkl file anywhere else.

In [ ]:
import pickle
pickle.dump(clf_pipeline, open('LogisticRegressionChurnModel.pkl','wb'))

In [ ]:
y_predict = clf_pipeline.predict(x)
print(y_predict)

[1 0 1 ... 1 1 0]


Once we have done with training a model we have to measure models performance. How our model performing. we do have different types of measure as per our need. For better Understanding you can watch : https://youtu.be/vdt0_y8CwG0?si=ug21iM9sNKNXWyv-

In [ ]:
print(confusion_matrix(y_test,y_predict))

[[1110  439]
 [ 115  446]]


In [ ]:
print(recall_score(y_test,y_predict))

0.7950089126559715


In [ ]:
print(precision_score(y_test,y_predict))

0.503954802259887


In [ ]:
print(roc_auc_score(y_test,y_predict))

0.7558001309567783


In [ ]:
print(classification_report(y_test,y_predict))

              precision    recall  f1-score   support

           0       0.91      0.72      0.80      1549
           1       0.50      0.80      0.62       561

    accuracy                           0.74      2110
   macro avg       0.71      0.76      0.71      2110
weighted avg       0.80      0.74      0.75      2110



Here how we can use our trained model from pickle file.

In [ ]:
loaded_pipeline = pickle.load(open('LogisticRegressionChurnModel.pkl', 'rb'))

sample_data = pd.DataFrame({
    "gender": ["Female"],
    "SeniorCitizen": [0],
    "Partner": ["Yes"],
    "Dependents": ["No"],
    "tenure": [1],
    "PhoneService": ["No"],
    "MultipleLines": ["No phone service"],
    "InternetService": ["DSL"],
    "OnlineSecurity": ["No"],
    "OnlineBackup": ["Yes"],
    "DeviceProtection": ["No"],
    "TechSupport": ["No"],
    "StreamingTV": ["No"],
    "StreamingMovies": ["No"],
    "Contract": ["Month-to-month"],
    "PaperlessBilling": ["Yes"],
    "PaymentMethod": ["Electronic check"],
    "MonthlyCharges": [29.85],
    "TotalCharges": [29.85]
})

prediction = loaded_pipeline.predict(sample_data)
probability = loaded_pipeline.predict_proba(sample_data)

print(f"Prediction: {prediction[0]}")
print(f"Probability of Churn: {probability[0][1]:.2%}")

Prediction: 1
Probability of Churn: 81.84%
